# 02 Pipeline
While FabricOps also supports reading from and writing to Fabric Warehouses when the pipeline requires Warehouse-based sources or targets.
- This workflow is optimized for Lakehouse to Lakehouse data movement using PySpark so pipelines can make use of distributed and parallel processing.
- A Lakehouse reads Parquet/Delta files directly from OneLake via the optimized file system path, 
- while a Warehouse forces PySpark to pull data through a TDS/SQL proxy connector with extra serialization overhead.
- In short - PySpark reads and write to Lakehouse better than Warehouse


## Tested with FabricOps

This notebook template is maintained separately from FabricOps package releases. The table below records the FabricOps releases that have been manually tested with this template in Microsoft Fabric.

| FabricOps release  | Tested by | Date tested | 
|---|---|---| 
| v0.2.0 |  Voyce |6 Aug 2026 | 


## 1. Run `00_env_config`

Run the `00_env_config` notebook before continuing. It establishes the active environment, configured Fabric stores, schemas and runtime context used by the pipeline.

In [ ]:
%run 00_env_config

## 2. Import required functions

In [ ]:
from fabricops_kit import (
 # FabricOps v0.1.0 onwards
    read_lakehouse_csv,
    read_lakehouse_excel,
    read_lakehouse_parquet,
    read_lakehouse_table,
    read_warehouse_query,
    read_warehouse_table,
    write_lakehouse_table,
    write_warehouse_table,
 # FabricOps v0.2.0 onwards
    profile_dataframe,
    profile_frequency_distribution,
    profile_and_register_table,
 # Preview
    observe_table,
    check_schema,
    check_freshness,
    check_changes,
    check_dq,
    widget_select_data_contract,
    widget_view_catalogue,
)



## 3. Read and profile source tables

Choose the example that matches your source. Define the target, schema and table name once, then reuse the same values for both reading and profiling. Run only one of the following examples.

#### `read_lakehouse_excel`

Use this for reading a excel stored in a Fabric Lakehouse.

In [ ]:
# These helpers are thin FabricOps path wrappers around standard readers:
# Always set target explicitly so it is clear which configured Lakehouse/Warehouse is used.
# - Excel uses pandas.read_excel options, then converts to Spark DataFrame.

# Example EXCEL file from Source lakehouse
source_df = read_lakehouse_excel(
     "excel_file_demo.xlsx",
     target="source",
     sheet_name="products",
     spark_session=spark,
 )
display(source_df)
# Optional
display(profile_dataframe(source_df))
display(profile_frequency_distribution(source_df))


#### `read_lakehouse_parquet`

Use this for reading a parquet stored in a Fabric Lakehouse.

In [ ]:
# These helpers are thin FabricOps path wrappers around standard readers:
# Always set target explicitly so it is clear which configured Lakehouse/Warehouse is used.
# - Parquet uses Spark Parquet reader options.

# Example PARQUET file from Source lakehouse
source_df = read_lakehouse_parquet(
     "parquet_file_demo.parquet",
     target="source",
     spark_session=spark,
 )
display(source_df)
# Optional
display(profile_dataframe(source_df))
display(profile_frequency_distribution(source_df))


#### `read_lakehouse_csv`

Use this for reading a csv stored in a Fabric Lakehouse.

In [ ]:
# These helpers are thin FabricOps path wrappers around standard readers:
# Always set target explicitly so it is clear which configured Lakehouse/Warehouse is used.
# - CSV uses Spark CSV reader options.

# Example CSV file from Source lakehouse:
source_df = read_lakehouse_csv(
     "lakehouse_data_demo.csv",          # This is relative path after Files/ so if you have folders do it 
     target="source",       # This is the name of your lakehouse you defined in 00_env_config
     spark_session=spark,
     header=True,
     inferSchema=True,
 )
display(source_df)

In [ ]:
# These helpers are thin FabricOps path wrappers around standard readers:
# Always set target explicitly so it is clear which configured Lakehouse/Warehouse is used.
# - CSV uses Spark CSV reader options.

# Example CSV file from Source lakehouse:
source_df = read_lakehouse_csv(
     "warehouse_data_demo.csv",          # This is relative path after Files/ so if you have folders do it 
     target="source",       # This is the name of your lakehouse you defined in 00_env_config
     spark_session=spark,
     header=True,
     inferSchema=True,
 )
display(source_df)

#### Cheap observation before `read_lakehouse_table`

Use the same logical source identity configured by `00_env_config`. `observe_table()` runs before the full read and produces compact evidence; it is not a guardrail. Schema, freshness, and change checks belong before the expensive read when they can consume metadata or observation evidence. Row-level data-quality checks and profiling belong after the read.

**Governed recurring runs only:** first onboard the source with a full read and `profile_and_register_table()` so Catalogue/Profiled evidence exists. Governance must then author and activate schema, freshness, and source-change rules. Only subsequent governed runs use this cheap pre-read section; `observe_table()` intentionally fails rather than guessing observation columns when no active source-change rule exists.


In [ ]:
SOURCE_TARGET = "source"
SOURCE_SCHEMA = "dbo"
SOURCE_TABLE_NAME = "student_enrolment"

# Development defaults to current authoring Guardrails; Production remains read only.
source_validation = widget_select_data_contract(
    SOURCE_TABLE_NAME, target=SOURCE_TARGET, schema=SOURCE_SCHEMA,
)

# 1. Observe and persist compact evidence before the full source read.
observation_df = observe_table(
    target=SOURCE_TARGET,
    schema=SOURCE_SCHEMA,
    table_name=SOURCE_TABLE_NAME,
)
# 2. Judge physical schema and the already-collected observation evidence.
schema_result = check_schema(
    target=SOURCE_TARGET,
    schema=SOURCE_SCHEMA,
    table_name=SOURCE_TABLE_NAME,
)
freshness_result = check_freshness(
    observation_df,
)
changes_result = check_changes(
    observation_df,
)
if not all(result["can_continue"] for result in (schema_result, freshness_result, changes_result)):
    raise RuntimeError("A pre-read source guardrail blocked the pipeline.")

# 3. Read the complete source only after every cheap check allows continuation.
source_df = read_lakehouse_table(
    target=SOURCE_TARGET,
    schema=SOURCE_SCHEMA,
    table_name=SOURCE_TABLE_NAME,
    spark_session=spark,
)

# 4. Run row-level DQ guardrails and persist failed-row evidence.
dq_result = check_dq(
    source_df, SOURCE_TABLE_NAME, target=SOURCE_TARGET, schema=SOURCE_SCHEMA,
)
display(dq_result["summary"])
if not dq_result["can_continue"]:
    raise RuntimeError("A DQ guardrail blocked the pipeline.")

# 5. Profile/register only after row-level checks pass.
source_profile_df = profile_and_register_table(
    source_df,
    profile_role="source",
    target=SOURCE_TARGET,
    schema=SOURCE_SCHEMA,
    table_name=SOURCE_TABLE_NAME,
)


In [ ]:
#Opitional 
display(source_df)

In [ ]:
#Opitional 
display(source_profile_df)

#### `read_warehouse_table`

Use this for reading a complete table from a configured Fabric Warehouse. Think of this as select *

#### `profile_and_register_table` 
- profiles the DataFrame representing a physical source or target table and writes its observed schema, profile, and data Lineage into FabricOps metadata tables.


In [ ]:
SOURCE_TARGET = "product"
SOURCE_SCHEMA = "demo"
SOURCE_TABLE_NAME = "email_logs_dummy"

source_df = read_warehouse_table(
    target=SOURCE_TARGET,
    schema=SOURCE_SCHEMA,
    table_name=SOURCE_TABLE_NAME,
    spark_session=spark,
)

source_profile_df = profile_and_register_table(
    source_df,
    profile_role="source",
    target=SOURCE_TARGET,
    schema=SOURCE_SCHEMA,
    table_name=SOURCE_TABLE_NAME,
)


#### `read_warehouse_query`

Use this for passing a SQL query of a table from a configured Fabric Warehouse. 

- When working with a very large Warehouse table, edit the SQL query to select only the required columns or rows. 
- Take Note : A filtered, joined or aggregated query result should not be registered as the complete profile of a single physical source table.

#### `profile_and_register_table` 
- profiles the DataFrame representing a physical source or target table and writes its observed schema, profile, and data Lineage into FabricOps metadata tables.


In [ ]:
SOURCE_TARGET = "product"
SOURCE_SCHEMA = "demo"
SOURCE_TABLE_NAME = "email_logs_dummy"
SOURCE_QUERY = f"""
SELECT
    MESSAGE_ID,
    RECEIVED_DTM,
    SENDER_ADDRESS,
    RECIPIENT_ADDRESS,
    SUBJECT,
    STATUS,
    -- Convert RECEIVED_DTM from UTC to Singapore time
    CAST(
        RECEIVED_DTM AT TIME ZONE 'UTC'
        AT TIME ZONE 'Singapore Standard Time'
        AS datetime2(0)
    ) AS RECEIVED_DTM_SGT,
    CAST(
        RECEIVED_DTM AT TIME ZONE 'UTC'
        AT TIME ZONE 'Singapore Standard Time'
        AS date
    ) AS RECEIVED_DATE_SGT
FROM [{SOURCE_SCHEMA}].[{SOURCE_TABLE_NAME}]
WHERE STATUS IN ('Delivered', 'Resolved')
AND CAST(
            RECEIVED_DTM AT TIME ZONE 'UTC' AT TIME ZONE 'Singapore Standard Time'
            AS date
        ) >= '2026-01-01'
        AND CAST(
            RECEIVED_DTM AT TIME ZONE 'UTC' AT TIME ZONE 'Singapore Standard Time'
            AS date
        ) <= '2026-01-31'
"""

source_df = read_warehouse_query(
    SOURCE_QUERY,
    target=SOURCE_TARGET,
    spark_session=spark,
)

# This profile is valid because the query returns the complete physical table.
source_profile_df = profile_and_register_table(
    source_df,
    profile_role="source",
    target=SOURCE_TARGET,
    schema=SOURCE_SCHEMA,
    table_name=SOURCE_TABLE_NAME,
)


In [ ]:
#Opitional 
display(source_df)

In [ ]:
#Opitional 
display(source_profile_df)

## 4. Source guardrails

Keep the cost boundary explicit:

1. `observe_table()` collects cheap source evidence.
2. Schema, freshness, and source-change guardrails make pre-read decisions where their available metadata/evidence permits.
3. Only after those checks pass, read the complete source.
4. Run null, value, domain, uniqueness, and other row-level DQ checks against `source_df`.
5. Profile and register the source after row-level checks pass.

`observe_table()` is evidence collection, not a guardrail. `check_schema()` inspects physical schema cheaply, `check_freshness(observation_df)` judges the latest observed change value, and `check_changes(observation_df)` compares the current evidence with the previous snapshot and owns removal tombstones.


In [ ]:
# Planned for v0.3.0


## 5. User defined transformation

Perform the transformations required by your pipeline. Use Microsoft Fabric Copilot to assist with transformation logic where relevant.

In [ ]:
# Simple example transformation on source_df
# - Adds an `ingested_at_utc` timestamp column
# - Leaves all existing columns unchanged

from pyspark.sql.functions import current_timestamp

transformed_df = source_df.withColumn("ingested_at_utc", current_timestamp())

display(transformed_df)


Target publication follows one environment-aware rule: **Dev writes and profiles first, then runs guardrails. Prod runs guardrails first, then writes and profiles.**

The branches remain visible in each standard target example. Schema guardrails resolve the configured target identity in both environments, while row-level DQ guardrails evaluate the same DataFrame used at that point in the lifecycle.


In [ ]:
# Planned for v0.3.0


## 7. Write and profile the target table (Lakehouse)
Write the transformed DataFrame to its configured Fabric target. After the write succeeds, read the persisted table and profile the physical target that now exists.

- Example
- Our compute is p2 capacity 
- Driver 8 vCores, Number of Executors 1–9 of which one is used for orchestration
- With this we can attempt to run up to 64 spark jobs concurrently
- But this does not guarantee Fabric will allocate all 8 executors to us as dynamic allocation decides that based on task backlog, capacity availability and scaling delay

- Note that we have commented out the partion_by and repartion_by parameters? 
`Pro tip` You can use that to do parallel processing with pyspark its faster but takes up more compute , 
- `be sure to use it on a proper partion column like date not datetime `else you will end up with a `multiple small files` problem which end up making the write and table read extremely slow.

In [ ]:
DF = transformed_df
TARGET_TARGET = "unified"
TARGET_SCHEMA = "demo"
TARGET_TABLE_NAME = "Laptop_Inventory"

if ENV == "dev":
    # Development: write/profile first, then guardrails.
    write_lakehouse_table(
        df=DF,
        target=TARGET_TARGET,
        schema=TARGET_SCHEMA,
        table_name=TARGET_TABLE_NAME,
        mode="append",
    )
    target_df = read_lakehouse_table(
        target=TARGET_TARGET,
        schema=TARGET_SCHEMA,
        table_name=TARGET_TABLE_NAME,
        spark_session=spark,
    )
    target_profile_df = profile_and_register_table(
        target_df,
        profile_role="target",
        target=TARGET_TARGET,
        schema=TARGET_SCHEMA,
        table_name=TARGET_TABLE_NAME,
    )
    check_schema(TARGET_TABLE_NAME, target=TARGET_TARGET, schema=TARGET_SCHEMA)
    dq_result = check_dq(target_df, TARGET_TABLE_NAME, target=TARGET_TARGET, schema=TARGET_SCHEMA)
    display(dq_result["summary"])
elif ENV == "prod":
    # Production: guardrails first, then write/profile.
    check_schema(
        TARGET_TABLE_NAME,
        target=TARGET_TARGET,
        schema=TARGET_SCHEMA,
        dataframe=DF,
    )
    dq_result = check_dq(DF, TARGET_TABLE_NAME, target=TARGET_TARGET, schema=TARGET_SCHEMA)
    display(dq_result["summary"])
    if not dq_result["can_continue"]:
        raise RuntimeError("A DQ guardrail blocked the target write.")
    write_lakehouse_table(
        df=DF,
        target=TARGET_TARGET,
        schema=TARGET_SCHEMA,
        table_name=TARGET_TABLE_NAME,
        mode="append",
    )
    target_df = read_lakehouse_table(
        target=TARGET_TARGET,
        schema=TARGET_SCHEMA,
        table_name=TARGET_TABLE_NAME,
        spark_session=spark,
    )
    target_profile_df = profile_and_register_table(
        target_df,
        profile_role="target",
        target=TARGET_TARGET,
        schema=TARGET_SCHEMA,
        table_name=TARGET_TABLE_NAME,
    )
else:
    raise ValueError(f"Unsupported pipeline environment: {ENV!r}. Expected 'dev' or 'prod'.")


In [ ]:
# Optional : 
display(target_df)

In [ ]:
# Optional :    
display(target_profile_df)

### Alternative 7. Write and profile the target table (Warehouse)
Write the transformed DataFrame to its configured Fabric target. After the write succeeds, read the persisted table and profile the physical target that now exists.

- Example
- Our compute is p2 capacity 
- Driver 8 vCores, Number of Executors 1–9 of which one is used for orchestration
- With this we can attempt to run up to 64 spark jobs concurrently
- But this does not guarantee Fabric will allocate all 8 executors to us as dynamic allocation decides that based on task backlog, capacity availability and scaling delay

- Note that we have commented out the partion_by and repartion_by parameters? 
`Pro tip` You can use that to do parallel processing with pyspark its faster but takes up more compute , 
- `be sure to use it on a proper partion column like date not datetime `else you will end up with a `multiple small files` problem which end up making the write and table read extremely slow.

In [ ]:
DF = source_df
TARGET_TARGET = "product"
TARGET_SCHEMA = "demo"
TARGET_TABLE_NAME = "email_logs_dummy"

if ENV == "dev":
    # Development: write/profile first, then guardrails.
    write_warehouse_table(
        df=DF,
        target=TARGET_TARGET,
        schema=TARGET_SCHEMA,
        table_name=TARGET_TABLE_NAME,
        mode="append",
    )
    target_df = read_warehouse_table(
        target=TARGET_TARGET,
        schema=TARGET_SCHEMA,
        table_name=TARGET_TABLE_NAME,
        spark_session=spark,
    )
    target_profile_df = profile_and_register_table(
        target_df,
        profile_role="target",
        target=TARGET_TARGET,
        schema=TARGET_SCHEMA,
        table_name=TARGET_TABLE_NAME,
    )
    check_schema(TARGET_TABLE_NAME, target=TARGET_TARGET, schema=TARGET_SCHEMA)
    dq_result = check_dq(target_df, TARGET_TABLE_NAME, target=TARGET_TARGET, schema=TARGET_SCHEMA)
    display(dq_result["summary"])
elif ENV == "prod":
    # Production: guardrails first, then write/profile.
    check_schema(
        TARGET_TABLE_NAME,
        target=TARGET_TARGET,
        schema=TARGET_SCHEMA,
        dataframe=DF,
    )
    dq_result = check_dq(DF, TARGET_TABLE_NAME, target=TARGET_TARGET, schema=TARGET_SCHEMA)
    display(dq_result["summary"])
    if not dq_result["can_continue"]:
        raise RuntimeError("A DQ guardrail blocked the target write.")
    write_warehouse_table(
        df=DF,
        target=TARGET_TARGET,
        schema=TARGET_SCHEMA,
        table_name=TARGET_TABLE_NAME,
        mode="append",
    )
    target_df = read_warehouse_table(
        target=TARGET_TARGET,
        schema=TARGET_SCHEMA,
        table_name=TARGET_TABLE_NAME,
        spark_session=spark,
    )
    target_profile_df = profile_and_register_table(
        target_df,
        profile_role="target",
        target=TARGET_TARGET,
        schema=TARGET_SCHEMA,
        table_name=TARGET_TABLE_NAME,
    )
else:
    raise ValueError(f"Unsupported pipeline environment: {ENV!r}. Expected 'dev' or 'prod'.")


## 8. Review this pipeline's Data Contract evidence

This technical self-review is restricted to the active environment, current Fabric workspace, and current notebook lineage. The pipeline author can review only datasets that this notebook registered or participated in through recorded lineage—not every registered dataset.


In [ ]:
pipeline_catalogue_view = widget_view_catalogue(
    mode="pipeline",
    target="metadata",
    spark_session=spark,
)


After changing the dataset selection above, rerun the review cell below. Guardrail Row Results / DQ Failure Evidence appears only when DQ rows failed for the selected run.


In [ ]:
views = pipeline_catalogue_view["get_views"]()

catalogue_df = views["catalogue"]
profile_df = views["profile"]
frequency_df = views["frequency"]
guardrail_results_df = views["guardrail_results"]
guardrail_row_results_df = views["guardrail_row_results"]


In [ ]:
display(catalogue_df)
display(profile_df)
display(frequency_df)
display(guardrail_results_df)
display(guardrail_row_results_df)
